# Module 05 — Neural Network Foundations
## 05-07: PyTorch Autograd

**Objective:** Understand how PyTorch builds and traverses computation graphs,
inspect `grad_fn` chains, and implement custom differentiable operations using
`torch.autograd.Function`.

**Prerequisites:** 05-06 (Backpropagation — chain rule, manual backprop);
01-03 (Calculus — gradient descent mechanics).


## Part 0 — Setup & Prerequisites

In **05-06** we implemented backpropagation entirely by hand.
PyTorch's autograd engine does exactly the same computation automatically,
by recording every operation in a **computation graph** and replaying the
chain rule in reverse during `.backward()`.

This notebook opens the autograd black box:
1. How tensors record their computational history (`requires_grad`, `grad_fn`, `is_leaf`).
2. How `.backward()` traverses the graph and accumulates ``.grad`` on leaf tensors.
3. How to **retain** intermediate gradients with `retain_grad()` and `register_hook()`.
4. How to write custom differentiable layers via `torch.autograd.Function`.
5. Advanced topics: `torch.no_grad()`, `detach()`, in-place pitfalls,
   higher-order gradients, and Jacobian computation.

> **Scope note:** We treat the built-in backprop for `nn.Linear`, `nn.Conv2d`, etc.
> as already verified in **05-06**. Here we focus on the graph-tracking machinery itself.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler

import random
import torch.nn.functional as F
from typing import Callable
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA:    {torch.version.cuda}")
    print(f"GPU:     {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Synthetic regression dataset (Part 4 training demo)
N_SAMPLES    = 400
N_FEATURES   = 10

# Architecture for training demo
HIDDEN_DIM   = 32
NUM_LAYERS   = 2
OUTPUT_DIM   = 1

# Training
BATCH_SIZE   = 32
NUM_EPOCHS   = 25
LEARNING_RATE = 1e-3

# gradcheck tolerance (float64 required)
GRADCHECK_EPS = 1e-6
GRADCHECK_ATOL = 1e-4


### Data — Synthetic Regression

We generate a small synthetic regression dataset used in Part 4 to compare
`CustomMLP` (using our hand-written autograd Functions) against `StandardMLP`
(using built-in `torch.relu`). Parts 1–3 use only toy tensors.


In [ ]:
# ── Synthetic regression dataset ─────────────────────────────────────────────
X_raw, y_raw = make_regression(
    n_samples    = N_SAMPLES,
    n_features   = N_FEATURES,
    n_informative = 8,
    noise        = 15.0,
    random_state = SEED,
)
y_raw = y_raw.reshape(-1, 1)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_sc = scaler_X.fit_transform(X_raw).astype(np.float32)
y_sc = scaler_y.fit_transform(y_raw).astype(np.float32)

# 80/10/10 split using random_split (DL convention)
generator = torch.Generator().manual_seed(SEED)
full_ds   = TensorDataset(torch.tensor(X_sc), torch.tensor(y_sc))
n_train   = int(0.8 * N_SAMPLES)
n_val     = int(0.1 * N_SAMPLES)
n_test    = N_SAMPLES - n_train - n_val
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val, n_test], generator=generator
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
print(f"Train: {n_train}  Val: {n_val}  Test: {n_test}")


---
## Part 1 — Computation Graph Internals

### How PyTorch Tracks Operations

Every time you apply an operation to a tensor with `requires_grad=True`,
PyTorch creates a **Function node** in a directed acyclic graph (DAG).
The node stores:

- **References to input tensors** (or their saved values) needed for the backward pass.
- **A backward function** (`grad_fn`) that knows how to compute $\partial\mathcal{L}/\partial\text{input}$ given $\partial\mathcal{L}/\partial\text{output}$.

When you call `.backward()` on the final scalar loss, PyTorch performs a
**reverse topological traversal** of this DAG, calling each `grad_fn` in turn and
accumulating gradients into `.grad` on leaf tensors.

| Property | Leaf tensor | Non-leaf tensor |
|---|---|---|
| `requires_grad` | True (if explicitly set) | True (if any parent has `requires_grad=True`) |
| `is_leaf` | **True** | False |
| `grad_fn` | `None` | Points to the operation that created it |
| `.grad` after backward | Stored | **Discarded** (use `retain_grad()` to keep) |


In [ ]:
# ── Building and inspecting the computation graph ────────────────────────────

# A simple expression: z = sum(x^2)
x_a = torch.tensor([3.0, 4.0], requires_grad=True)
y_a = x_a ** 2           # PowBackward0
z_a = y_a.sum()          # SumBackward0

print("=== Tensor attributes ===")
print(f"  x_a.requires_grad : {x_a.requires_grad}")
print(f"  x_a.is_leaf        : {x_a.is_leaf}")
print(f"  x_a.grad_fn        : {x_a.grad_fn}")    # None — leaf has no grad_fn
print(f"  y_a.is_leaf        : {y_a.is_leaf}")
print(f"  y_a.grad_fn        : {y_a.grad_fn}")    # PowBackward0
print(f"  z_a.grad_fn        : {z_a.grad_fn}")    # SumBackward0


def trace_grad_fn(
    grad_fn: object,
    depth: int = 0,
    max_depth: int = 6,
) -> None:
    '''Recursively print the grad_fn ancestry of a tensor.

    Traverses the next_functions chain (inputs of each Function node),
    printing each node type with increasing indentation.

    Args:
        grad_fn: The grad_fn of a non-leaf tensor (or None for leaves).
        depth: Current recursion depth (controls indentation).
        max_depth: Stop traversal beyond this depth.
    '''
    if depth > max_depth or grad_fn is None:
        return
    indent = "  " * depth
    print(f"{indent}{type(grad_fn).__name__}")
    for next_fn, _ in grad_fn.next_functions:
        trace_grad_fn(next_fn, depth + 1, max_depth)


print("\n=== grad_fn chain: z_a = (x_a**2).sum() ===")
trace_grad_fn(z_a.grad_fn)

# Run backward and inspect gradients
z_a.backward()
print(f"\nx_a.grad: {x_a.grad}  (expected 2*x = [6., 8.])")
print(f"y_a.grad:  {y_a.grad}  (None — intermediate, no retain_grad)")

# ── Gradient accumulation: the reason for zero_grad() ────────────────────────
print("\n=== Gradient accumulation (why we call optimizer.zero_grad()) ===")
w_acc = torch.tensor([2.0], requires_grad=True)
for step in range(4):
    loss_acc = (w_acc * 3.0).sum()
    loss_acc.backward()
    print(f"  Step {step + 1}: w_acc.grad = {w_acc.grad.item():.1f}  (grad += 3.0 each step)")
w_acc.grad.zero_()
print(f"  After .zero_(): w_acc.grad = {w_acc.grad.item():.1f}")
print("  => Always call zero_grad() before each backward to avoid accumulation.")

# ── More complex graph: MLP-like chain ───────────────────────────────────────
print("\n=== Graph for: y = relu(x @ W + b).sum() ===")
x_mlp = torch.randn(4, 3, requires_grad=False)   # input: not tracked
W_mlp = torch.randn(3, 5, requires_grad=True)    # weight: tracked
b_mlp = torch.zeros(5,     requires_grad=True)   # bias: tracked
h_mlp = torch.relu(x_mlp @ W_mlp + b_mlp)
y_mlp = h_mlp.sum()
trace_grad_fn(y_mlp.grad_fn, max_depth=4)


### 1.2 Leaf vs Non-Leaf Tensors in Detail

The leaf/non-leaf distinction has two practical implications:

1. **`.grad` is only populated for leaves** after `.backward()`. Non-leaf
   intermediate tensors discard their gradients immediately to free memory
   (unless you call `retain_grad()` or `register_hook()` before the backward pass).
2. **`nn.Parameter` wraps a leaf tensor** — that is why model weights accumulate
   gradients and can be updated by the optimizer.


In [ ]:
# ── Leaf vs non-leaf: detailed exploration ───────────────────────────────────
print("=== Leaf/non-leaf rules ===")

# (1) User-created tensor with requires_grad=True -> leaf
leaf_rg   = torch.randn(3, requires_grad=True)
print(f"randn(requires_grad=True)  : is_leaf={leaf_rg.is_leaf}")

# (2) User-created tensor with requires_grad=False -> also a leaf (but no grad)
leaf_norg = torch.randn(3, requires_grad=False)
print(f"randn(requires_grad=False) : is_leaf={leaf_norg.is_leaf}")

# (3) Computed tensor -> non-leaf
non_leaf  = leaf_rg * 2.0
print(f"leaf_rg * 2.0 (computed)   : is_leaf={non_leaf.is_leaf}, "
      f"grad_fn={non_leaf.grad_fn}")

# ── nn.Linear parameters are leaves ──────────────────────────────────────────
print("\n=== nn.Module parameters are leaf tensors ===")
layer = nn.Linear(4, 3)
for name, param in layer.named_parameters():
    print(f"  {name:<12s}: is_leaf={param.is_leaf}  "
          f"requires_grad={param.requires_grad}  shape={tuple(param.shape)}")

# ── detach() creates a new leaf that shares storage ───────────────────────────
print("\n=== detach(): stop gradient at a tensor ===")
x_det  = torch.randn(3, requires_grad=True)
h_det  = torch.relu(x_det)          # non-leaf
h_stop = h_det.detach()             # new leaf, shares data, no grad
print(f"h_det  : is_leaf={h_det.is_leaf}   grad_fn={h_det.grad_fn}")
print(f"h_stop : is_leaf={h_stop.is_leaf}  grad_fn={h_stop.grad_fn}  "
      f"requires_grad={h_stop.requires_grad}")
# Modifying h_stop does NOT affect the gradient of h_det (they share data,
# but h_stop has no grad_fn so the graph is cut)

# ── requires_grad_() toggles gradient tracking on a leaf ─────────────────────
print("\n=== requires_grad_() toggle ===")
x_tog = torch.randn(3)
print(f"Before: requires_grad={x_tog.requires_grad}")
x_tog.requires_grad_(True)
print(f"After True: requires_grad={x_tog.requires_grad}")
x_tog.requires_grad_(False)
print(f"After False: requires_grad={x_tog.requires_grad}")

# ── nn.Parameter: leaf tensor that registers as a model parameter ─────────────
p = nn.Parameter(torch.randn(3))
print(f"\nnn.Parameter: is_leaf={p.is_leaf}  requires_grad={p.requires_grad}")

# ── Gradient only flows through requires_grad=True tensors ───────────────────
print("\n=== Gradient blocking by requires_grad=False ===")
x_blk = torch.randn(3, requires_grad=True)
c_blk = torch.randn(3, requires_grad=False)  # constant
y_blk = (x_blk * c_blk).sum()
y_blk.backward()
print(f"x_blk.grad: {x_blk.grad}  (= c_blk: gradient flows through x)")
print(f"c_blk.grad: {c_blk.grad}  (None: requires_grad=False)")


### 1.3 Retaining Intermediate Gradients & Tensor Hooks

By default autograd discards the gradient of non-leaf tensors after `.backward()`
to free memory.  Two mechanisms let you inspect them:

- **`tensor.retain_grad()`** — call before the backward pass; the tensor's `.grad`
  will be populated after backward (at the cost of keeping the gradient in memory).
- **`tensor.register_hook(fn)`** — registers a callback invoked with the gradient
  *during* the backward pass.  Useful for debugging, gradient logging, or gradient clipping.


In [ ]:
# ── retain_grad(): keep a non-leaf's gradient after backward ─────────────────
print("=== retain_grad() ===")
x_rg  = torch.randn(4, requires_grad=True)
h_rg  = torch.relu(x_rg)           # non-leaf
y_rg  = (h_rg ** 2).sum()

y_rg.backward(retain_graph=True)
print(f"h_rg.grad WITHOUT retain_grad() : {h_rg.grad}")   # None

# Reset, now call retain_grad() before backward
x_rg2 = torch.randn(4, requires_grad=True)
h_rg2 = torch.relu(x_rg2)
h_rg2.retain_grad()
y_rg2 = (h_rg2 ** 2).sum()
y_rg2.backward()
print(f"h_rg2.grad WITH retain_grad()   : {h_rg2.grad}")  # = 2*h_rg2

# ── register_hook(): intercept a gradient mid-backward ───────────────────────
print("\n=== register_hook() ===")
grad_log: list[torch.Tensor] = []

x_hook = torch.randn(4, requires_grad=True)
h_hook = torch.relu(x_hook)
handle = h_hook.register_hook(lambda g: grad_log.append(g.clone()))
y_hook = h_hook.sum()
y_hook.backward()
handle.remove()   # always remove hooks after use to avoid memory leaks
print(f"Gradient intercepted by hook: {grad_log[0]}")
print(f"h_hook.grad (still None):     {h_hook.grad}")

# ── retain_graph=True: re-using a graph for multiple backward passes ──────────
print("\n=== retain_graph=True: call backward twice on the same graph ===")
x_rg3 = torch.tensor([2.0, 3.0], requires_grad=True)
y_rg3 = (x_rg3 ** 2).sum()   # z = 9 + 4 = 13

y_rg3.backward(retain_graph=True)
first_grad = x_rg3.grad.clone()
x_rg3.grad.zero_()

y_rg3.backward()              # second backward: graph is intact
second_grad = x_rg3.grad.clone()

print(f"First  backward: {first_grad}")    # [4., 6.]
print(f"Second backward: {second_grad}")   # [4., 6.] — graph correctly reused
print("Without retain_graph=True, the second call raises RuntimeError.")

# ── Gradient magnitude monitoring with a hook ─────────────────────────────────
print("\n=== Gradient monitoring hook on a model parameter ===")
tiny_net = nn.Linear(4, 1)
grad_norms_hook: list[float] = []

def _log_grad_norm(grad: torch.Tensor) -> None:
    '''Append the Frobenius norm of a gradient tensor to grad_norms_hook.'''
    grad_norms_hook.append(float(grad.norm().item()))

hook_handle = tiny_net.weight.register_hook(_log_grad_norm)
x_tiny = torch.randn(8, 4)
y_tiny = torch.randn(8, 1)
for _ in range(5):
    loss_t = nn.MSELoss()(tiny_net(x_tiny), y_tiny)
    loss_t.backward()
    tiny_net.zero_grad()
hook_handle.remove()
print(f"Gradient norms logged across 5 backward passes: "
      f"{[f'{n:.4f}' for n in grad_norms_hook]}")


### 1.4 Module-Level Backward Hooks

Beyond tensor-level `register_hook()`, `nn.Module` exposes
`register_full_backward_hook(fn)` which fires during the backward pass
for the **entire layer**, receiving `(module, grad_input, grad_output)`.  This is the standard way to monitor or modify gradient flow without touching
individual tensor nodes.


In [ ]:
# ── Module-level backward hooks: register_full_backward_hook ─────────────────
# While register_hook() works on individual tensors, nn.Module provides
# register_full_backward_hook() to intercept gradients at the module level.
# The callback receives (module, grad_input, grad_output).

layer_grad_log: dict[str, list[float]] = {}

def make_grad_logger(name: str) -> Callable:
    '''Create a backward hook that logs the L2 norm of grad_output.

    Args:
        name: Label for this layer in layer_grad_log.

    Returns:
        A hook function compatible with register_full_backward_hook.
    '''
    def _hook(module: nn.Module,
              grad_input: tuple[torch.Tensor, ...],
              grad_output: tuple[torch.Tensor, ...]) -> None:
        '''Log the norm of the first grad_output tensor.'''
        norm_val = float(grad_output[0].norm().item()) if grad_output[0] is not None else 0.0
        layer_grad_log.setdefault(name, []).append(norm_val)
    return _hook


# Register hooks on all linear layers of custom_model
hook_handles: list = []
for lyr_idx, linear in enumerate(custom_model.linears):
    lbl = f"Linear_{lyr_idx + 1}"
    handle = linear.register_full_backward_hook(make_grad_logger(lbl))
    hook_handles.append(handle)

# Run N training steps and collect norms
N_HOOK_STEPS = 60
custom_model.train()
optimizer_hook = optim.Adam(custom_model.parameters(), lr=LEARNING_RATE)
criterion_hook = nn.MSELoss()

step_count = 0
for X_b, y_b in train_loader:
    if step_count >= N_HOOK_STEPS:
        break
    X_b, y_b = X_b.to(device), y_b.to(device)
    optimizer_hook.zero_grad()
    loss_h = criterion_hook(custom_model(X_b), y_b)
    loss_h.backward()
    optimizer_hook.step()
    step_count += 1

for handle in hook_handles:
    handle.remove()

# ── Plot gradient norms per layer ─────────────────────────────────────────────
fig_hooks, ax_hooks = plt.subplots(figsize=(10, 4))
colors_h = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for i, (lbl, norms) in enumerate(sorted(layer_grad_log.items())):
    ax_hooks.plot(range(len(norms)), norms,
                  color=colors_h[i % len(colors_h)], lw=1.5, label=lbl)
ax_hooks.set_xlabel("Training step", fontsize=11)
ax_hooks.set_ylabel("||grad_output||_2", fontsize=11)
ax_hooks.set_title("Gradient Norms per Layer (via register_full_backward_hook)",
                    fontsize=11, fontweight="bold")
ax_hooks.legend(fontsize=9)
ax_hooks.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Module backward hook summary:")
for lbl, norms in sorted(layer_grad_log.items()):
    print(f"  {lbl}: mean={float(np.mean(norms)):.5f}  "
          f"max={float(np.max(norms)):.5f}  steps={len(norms)}")
print("Gradient norms are typically larger near the output and smaller near the input.")


---
## Part 2 — Custom `torch.autograd.Function`

### Why Custom Functions?

PyTorch's built-in autograd handles most operations.  But you may need a custom
`Function` when:

- Implementing a **novel activation** not in `torch.nn`.
- Applying a **non-differentiable operation** and providing a surrogate gradient.
- Optimising the backward pass (e.g., recomputing rather than caching).
- Bridging with **non-PyTorch code** (e.g., custom CUDA kernels, NumPy).

### API

```python
class MyFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *inputs):
        # ctx.save_for_backward(*tensors)  — save tensors for backward
        # ctx.my_scalar = value            — save non-tensors
        return output

    @staticmethod
    def backward(ctx, *grad_outputs):
        # tensors = ctx.saved_tensors      — retrieve saved tensors
        # return one gradient per input (or None if no gradient needed)
        return grad_input_1, grad_input_2, ...

output = MyFn.apply(*inputs)   # call via .apply(), NOT MyFn(...)
```

`torch.autograd.gradcheck` numerically verifies that `backward` matches finite differences.
It requires **float64** inputs for sufficient numerical precision.


In [ ]:
# ── Custom ReLU: torch.autograd.Function ─────────────────────────────────────

class CustomReLUFn(torch.autograd.Function):
    '''ReLU implemented as a custom autograd Function.

    Forward:  A = max(0, Z)
    Backward: dZ = dA * (Z > 0)  [ReLU gate]

    Saves the input Z for use in the backward pass.
    '''

    @staticmethod
    def forward(ctx: torch.autograd.function.FunctionCtx,
                z: torch.Tensor) -> torch.Tensor:
        '''Apply ReLU; save input tensor for backward.

        Args:
            ctx: Context object for saving state.
            z: Pre-activation tensor of any shape.

        Returns:
            Activated tensor max(0, z) of same shape as z.
        '''
        ctx.save_for_backward(z)
        return z.clamp(min=0.0)

    @staticmethod
    def backward(ctx: torch.autograd.function.FunctionCtx,
                 grad_output: torch.Tensor) -> torch.Tensor:
        '''Pass gradient only where z > 0 (ReLU gate).

        Args:
            ctx: Context with saved tensors from forward.
            grad_output: Upstream gradient dL/dA of same shape as output.

        Returns:
            Gradient dL/dZ of same shape as input z.
        '''
        z, = ctx.saved_tensors
        grad_z = grad_output.clone()
        grad_z[z <= 0] = 0.0
        return grad_z


def custom_relu(z: torch.Tensor) -> torch.Tensor:
    '''Apply CustomReLUFn to tensor z.

    Args:
        z: Input tensor.

    Returns:
        Tensor with CustomReLU applied.
    '''
    return CustomReLUFn.apply(z)


# ── Numerical verification ────────────────────────────────────────────────────
x_relu = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0], requires_grad=True)
out_cr = custom_relu(x_relu)
out_cr.sum().backward()
grad_cr = x_relu.grad.clone()

x_relu.grad.zero_()
out_tr = torch.relu(x_relu)
out_tr.sum().backward()
grad_tr = x_relu.grad.clone()

print("Custom ReLU vs torch.relu:")
print(f"  Forward  max diff: {(out_cr - out_tr).abs().max().item():.2e}")
print(f"  Backward max diff: {(grad_cr - grad_tr).abs().max().item():.2e}")

# grad_fn chain shows our custom function in the graph
print(f"\ngrad_fn of custom_relu output: {out_cr.grad_fn}")
print(f"grad_fn of torch.relu  output: {out_tr.grad_fn}")

# ── gradcheck: rigorous numerical gradient verification ───────────────────────
# Note: gradcheck requires float64 for numerical precision
x_gc_relu = torch.randn(8, dtype=torch.float64, requires_grad=True)
gc_relu_pass = torch.autograd.gradcheck(
    CustomReLUFn.apply, x_gc_relu, eps=GRADCHECK_EPS, atol=GRADCHECK_ATOL,
)
print(f"\ngradcheck (CustomReLU):   {'PASS' if gc_relu_pass else 'FAIL'}")


In [ ]:
# ── Custom Sigmoid: saving the output (more efficient than saving input) ──────

class CustomSigmoidFn(torch.autograd.Function):
    '''Sigmoid implemented as a custom autograd Function.

    Forward:  sigma(x) = 1 / (1 + exp(-x))
    Backward: d_sigma/dx = sigma * (1 - sigma)

    Saves sigma (the output) rather than x, since the derivative is
    expressed more simply in terms of the output.
    '''

    @staticmethod
    def forward(ctx: torch.autograd.function.FunctionCtx,
                x: torch.Tensor) -> torch.Tensor:
        '''Compute sigmoid and save output for backward.

        Args:
            ctx: Context object.
            x: Input tensor of any shape.

        Returns:
            sigma(x) of same shape as x.
        '''
        sigma = torch.sigmoid(x)      # use built-in for numerical stability
        ctx.save_for_backward(sigma)  # save sigma, NOT x (see backward)
        return sigma

    @staticmethod
    def backward(ctx: torch.autograd.function.FunctionCtx,
                 grad_output: torch.Tensor) -> torch.Tensor:
        '''Compute gradient using the saved sigmoid output.

        The derivative sigma*(1-sigma) is computed from the saved output,
        avoiding a second evaluation of exp(-x).

        Args:
            ctx: Context with saved sigmoid output.
            grad_output: Upstream gradient dL/d_sigma.

        Returns:
            Gradient dL/dx of same shape as x.
        '''
        sigma, = ctx.saved_tensors
        return grad_output * sigma * (1.0 - sigma)


def custom_sigmoid(x: torch.Tensor) -> torch.Tensor:
    '''Apply CustomSigmoidFn to tensor x.

    Args:
        x: Input tensor.

    Returns:
        Tensor with CustomSigmoid applied.
    '''
    return CustomSigmoidFn.apply(x)


# ── Verification ──────────────────────────────────────────────────────────────
x_sig = torch.randn(10, requires_grad=True)
fwd_diff = (custom_sigmoid(x_sig) - torch.sigmoid(x_sig)).abs().max().item()
print(f"Custom Sigmoid forward max diff: {fwd_diff:.2e}")

x_gc_sig = torch.randn(8, dtype=torch.float64, requires_grad=True)
gc_sig_pass = torch.autograd.gradcheck(
    CustomSigmoidFn.apply, x_gc_sig, eps=GRADCHECK_EPS, atol=GRADCHECK_ATOL,
)
print(f"gradcheck (CustomSigmoid):   {'PASS' if gc_sig_pass else 'FAIL'}")

# Show that saving sigma (output) instead of x (input) is correct
x_demo = torch.tensor([0.0, 1.0, -1.0], requires_grad=True)
y_demo = custom_sigmoid(x_demo).sum()
y_demo.backward()
sigma_val = torch.sigmoid(x_demo.detach())
expected_grad = sigma_val * (1.0 - sigma_val)
print(f"Analytical gradient: {expected_grad}")
print(f"Autograd gradient:   {x_demo.grad}")
print(f"Match: {(x_demo.grad - expected_grad).abs().max().item() < 1e-6}")


In [ ]:
# ── Custom MSE Loss Function ──────────────────────────────────────────────────

class CustomMSEFn(torch.autograd.Function):
    '''MSE loss implemented as a custom autograd Function.

    Forward:  L = mean((y_hat - y_true)^2)
    Backward: dL/d_y_hat = 2 * (y_hat - y_true) / n

    y_true is treated as a constant (no gradient returned for it).
    '''

    @staticmethod
    def forward(ctx: torch.autograd.function.FunctionCtx,
                y_hat: torch.Tensor,
                y_true: torch.Tensor) -> torch.Tensor:
        '''Compute MSE loss and save the residuals for backward.

        Args:
            ctx: Context object.
            y_hat: Predictions of shape (n, 1) or (n,).
            y_true: Ground-truth targets of same shape as y_hat.

        Returns:
            Scalar MSE loss tensor.
        '''
        diff = y_hat - y_true
        ctx.save_for_backward(diff)
        ctx.n = diff.numel()
        return (diff ** 2).mean()

    @staticmethod
    def backward(ctx: torch.autograd.function.FunctionCtx,
                 grad_output: torch.Tensor) -> tuple[torch.Tensor, None]:
        '''Compute gradient w.r.t. y_hat; return None for y_true.

        Args:
            ctx: Context with saved diff tensor.
            grad_output: Upstream scalar gradient (typically 1.0 from loss.backward()).

        Returns:
            Tuple of (gradient w.r.t. y_hat, None for y_true).
        '''
        diff, = ctx.saved_tensors
        n     = ctx.n
        return grad_output * 2.0 * diff / n, None


def custom_mse(
    y_hat: torch.Tensor,
    y_true: torch.Tensor,
) -> torch.Tensor:
    '''Compute MSE loss using CustomMSEFn.

    Args:
        y_hat: Predictions tensor.
        y_true: Target tensor (no gradient computed for this).

    Returns:
        Scalar MSE loss.
    '''
    return CustomMSEFn.apply(y_hat, y_true)


# ── Verify against nn.MSELoss ─────────────────────────────────────────────────
y_hat_v  = torch.randn(8, 1, requires_grad=True)
y_true_v = torch.randn(8, 1)

loss_custom = custom_mse(y_hat_v, y_true_v)
loss_torch  = nn.MSELoss()(y_hat_v, y_true_v)
print(f"Custom MSE loss:      {loss_custom.item():.6f}")
print(f"nn.MSELoss:           {loss_torch.item():.6f}")
print(f"Difference:           {abs(loss_custom.item() - loss_torch.item()):.2e}")

loss_custom.backward()
grad_custom_mse = y_hat_v.grad.clone()
y_hat_v.grad.zero_()
loss_torch.backward()
grad_torch_mse = y_hat_v.grad.clone()
print(f"Gradient max diff:    {(grad_custom_mse - grad_torch_mse).abs().max().item():.2e}")

# gradcheck: pass only y_hat (y_true has no gradient)
y_hat_gc_mse  = torch.randn(6, 1, dtype=torch.float64, requires_grad=True)
y_true_gc_mse = torch.randn(6, 1, dtype=torch.float64)
gc_mse_pass = torch.autograd.gradcheck(
    lambda yh: CustomMSEFn.apply(yh, y_true_gc_mse),
    y_hat_gc_mse,
    eps=GRADCHECK_EPS,
    atol=GRADCHECK_ATOL,
)
print(f"gradcheck (CustomMSE):   {'PASS' if gc_mse_pass else 'FAIL'}")


### 2.4 Custom Leaky ReLU — Saving Non-Tensor Context

When a custom `Function` needs to use a **Python scalar** (e.g. a hyperparameter)
in the backward pass, it cannot call `ctx.save_for_backward()` (tensors only).
Instead, store it directly on the context object: `ctx.alpha = alpha`.
The backward must return `None` for this input since there is no gradient for a
non-differentiable Python scalar.


In [ ]:
# ── Custom Leaky ReLU: non-tensor context values in backward ─────────────────
# Leaky ReLU: f(x) = x if x > 0, else alpha*x.
# df/dx = 1 if x > 0, else alpha.
# The slope alpha is a Python float (hyperparameter), saved via ctx.alpha.

class CustomLeakyReLUFn(torch.autograd.Function):
    '''Leaky ReLU with configurable negative slope, as a custom autograd Function.

    Forward:  f(x) = x  if x > 0, else alpha*x
    Backward: df/dx = 1 if x > 0, else alpha

    alpha is stored on the context as a Python scalar (not a tensor).
    '''

    @staticmethod
    def forward(ctx: torch.autograd.function.FunctionCtx,
                x: torch.Tensor,
                alpha: float) -> torch.Tensor:
        '''Apply leaky ReLU; save mask and alpha for backward.

        Args:
            ctx: Context for saving state.
            x: Input tensor of any shape.
            alpha: Negative slope coefficient (not a tensor).

        Returns:
            Activated tensor of same shape as x.
        '''
        ctx.save_for_backward(x)
        ctx.alpha = alpha       # save non-tensor as attribute
        return torch.where(x > 0, x, alpha * x)

    @staticmethod
    def backward(ctx: torch.autograd.function.FunctionCtx,
                 grad_output: torch.Tensor) -> tuple[torch.Tensor, None]:
        '''Backpropagate through leaky ReLU.

        Args:
            ctx: Context with saved x and alpha.
            grad_output: Upstream gradient.

        Returns:
            Tuple of (gradient w.r.t. x, None for alpha — not a tensor).
        '''
        x, = ctx.saved_tensors
        alpha = ctx.alpha
        gate  = torch.where(x > 0,
                            torch.ones_like(x),
                            torch.full_like(x, alpha))
        return grad_output * gate, None   # None: alpha has no gradient


def custom_leaky_relu(x: torch.Tensor, alpha: float = 0.1) -> torch.Tensor:
    '''Apply CustomLeakyReLUFn with the given negative slope.

    Args:
        x: Input tensor.
        alpha: Negative slope (default 0.1).

    Returns:
        Tensor with leaky ReLU applied.
    '''
    return CustomLeakyReLUFn.apply(x, alpha)


# ── Verify against F.leaky_relu ───────────────────────────────────────────────

x_lr = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
alpha_test = 0.2

out_custom_lr = custom_leaky_relu(x_lr, alpha_test)
out_builtin   = F.leaky_relu(x_lr, negative_slope=alpha_test)
fwd_diff_lr   = (out_custom_lr - out_builtin).abs().max().item()

out_custom_lr.sum().backward(retain_graph=True)
grad_custom_lr = x_lr.grad.clone()
x_lr.grad.zero_()

out_builtin.sum().backward()
grad_builtin_lr = x_lr.grad.clone()

print("Custom LeakyReLU vs F.leaky_relu (alpha=0.2):")
print(f"  Forward  max diff: {fwd_diff_lr:.2e}")
print(f"  Backward max diff: {(grad_custom_lr - grad_builtin_lr).abs().max().item():.2e}")

# gradcheck
x_gc_lr = torch.randn(8, dtype=torch.float64, requires_grad=True)
gc_lr_pass = torch.autograd.gradcheck(
    lambda x: CustomLeakyReLUFn.apply(x, 0.2),
    x_gc_lr,
    eps=GRADCHECK_EPS, atol=GRADCHECK_ATOL,
)
print(f"  gradcheck: {'PASS' if gc_lr_pass else 'FAIL'}")

# ── Negative slope sweep: how alpha affects gradient magnitude ─────────────────
alphas    = [0.0, 0.01, 0.1, 0.2, 0.5, 1.0]
fwd_means = []
bwd_means = []

for a in alphas:
    x_sw = torch.randn(1000, requires_grad=True)
    out_sw = custom_leaky_relu(x_sw, alpha=a)
    fwd_means.append(float(out_sw.mean().item()))
    out_sw.sum().backward()
    bwd_means.append(float(x_sw.grad.mean().item()))
    x_sw.grad.zero_()

fig_lr, axes_lr = plt.subplots(1, 2, figsize=(12, 4))
axes_lr[0].bar([str(a) for a in alphas], fwd_means, color="#1f77b4", edgecolor="k", lw=0.7)
axes_lr[0].set_xlabel("alpha (negative slope)", fontsize=11)
axes_lr[0].set_ylabel("Mean output (1000 random inputs)", fontsize=11)
axes_lr[0].set_title("Forward: Mean Activation vs alpha", fontsize=11, fontweight="bold")
axes_lr[0].grid(axis="y", alpha=0.3)

axes_lr[1].bar([str(a) for a in alphas], bwd_means, color="#d62728", edgecolor="k", lw=0.7)
axes_lr[1].set_xlabel("alpha (negative slope)", fontsize=11)
axes_lr[1].set_ylabel("Mean gradient", fontsize=11)
axes_lr[1].set_title("Backward: Mean Gradient vs alpha", fontsize=11, fontweight="bold")
axes_lr[1].grid(axis="y", alpha=0.3)
axes_lr[1].axhline(0.5, color="k", lw=1, ls="--", label="Expected (50% pos, 50% neg)")
axes_lr[1].legend(fontsize=8)
plt.suptitle("Leaky ReLU: Effect of Negative Slope on Activations and Gradients",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print("alpha=0.0 is standard ReLU (dead neurons possible).")
print("alpha=1.0 is linear (no non-linearity, no dead neurons).")


---
## Part 3 — Advanced Autograd Topics

This section covers four practical topics that every PyTorch practitioner encounters:

| Topic | Key API | Typical Use |
|-------|---------|-------------|
| `torch.no_grad()` | Context manager | Inference, metric computation |
| `.detach()` | Tensor method | Stop-gradient in target networks, GANs |
| In-place operations | `+=`, `.add_()` | Bug-prone — avoid on tensors with `grad_fn` |
| Higher-order gradients | `create_graph=True` | MAML, gradient penalty, second-order optimisers |


In [ ]:
# ── torch.no_grad() and detach() ─────────────────────────────────────────────

# ── (1) torch.no_grad(): disable graph construction ───────────────────────────
print("=== torch.no_grad() ===")
x_ng = torch.randn(4, 4, requires_grad=True)
W_ng = torch.randn(4, 4, requires_grad=True)

with torch.no_grad():
    y_ng = x_ng @ W_ng
print(f"  y (inside no_grad) .grad_fn:       {y_ng.grad_fn}")      # None
print(f"  y (inside no_grad) .requires_grad: {y_ng.requires_grad}") # False

y_ng_grad = x_ng @ W_ng
print(f"  y (outside no_grad).grad_fn:       {y_ng_grad.grad_fn}")  # MatMulBackward

# Timing comparison: no_grad is faster and uses less memory
n_iters = 1000
x_time = torch.randn(128, 128, requires_grad=True)
W_time = torch.randn(128, 128, requires_grad=True)

t0 = time.perf_counter()
for _ in range(n_iters):
    _ = x_time @ W_time
t_with = time.perf_counter() - t0

t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_iters):
        _ = x_time @ W_time
t_no = time.perf_counter() - t0
print(f"  With grad:    {t_with * 1000:.1f} ms ({n_iters} iters)")
print(f"  no_grad:      {t_no   * 1000:.1f} ms ({n_iters} iters)")
print(f"  Speedup:      {t_with / t_no:.2f}x")

# ── (2) detach(): cut gradient flow at a specific tensor ─────────────────────
print("\n=== detach() — stop-gradient trick ===")
# Common in: RL target networks, contrastive learning, GAN training
x_dg = torch.randn(5, requires_grad=True)
online_net = nn.Linear(5, 3)
target_net = nn.Linear(5, 3)

online_out = online_net(x_dg)
target_out = target_net(x_dg).detach()  # no gradient flows through target
loss_dg    = nn.MSELoss()(online_out, target_out)
loss_dg.backward()

# online_net.weight.grad: populated (gradient flows back through online_out)
# target_net.weight.grad: None (detach() blocked the graph)
print(f"  online_net.weight.grad is not None: {online_net.weight.grad is not None}")
print(f"  target_net.weight.grad is not None: {target_net.weight.grad is not None}")
print("  => detach() is the stop-gradient operator used in BYOL, DQN target nets, etc.")

# ── (3) @torch.no_grad() as a function decorator ─────────────────────────────
print("\n=== @torch.no_grad() decorator ===")

@torch.no_grad()
def inference(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    '''Run model inference without building the computation graph.

    Args:
        model: PyTorch model.
        x: Input tensor.

    Returns:
        Model output (no grad_fn).
    '''
    return model(x)

x_inf = torch.randn(4, 5)
out_inf = inference(online_net, x_inf)
print(f"  Inference output .grad_fn: {out_inf.grad_fn}")  # None


### 3.2 In-Place Operation Pitfalls

In-place ops (e.g. `tensor += value`, `.add_()`) modify the tensor's storage directly.
PyTorch tracks this with an internal **version counter**.  If an in-place op increments
the counter after a node has cached a reference to that tensor, the backward pass detects
the modification and raises a `RuntimeError` to prevent silent incorrect gradients.

There is one silent failure mode: modifying `tensor.data` directly bypasses the version
counter and produces wrong gradients **without any error**.  Always prefer out-of-place ops.


In [ ]:
# ── In-place operation pitfalls ───────────────────────────────────────────────
# In-place ops modify the tensor data directly, incrementing a version counter.
# If the modified tensor was used in a graph node, the saved reference becomes
# invalid during backward, causing a RuntimeError.

print("=== In-place operations ===")

# ── (1) Safe: in-place on a leaf NOT yet part of a graph ─────────────────────
x_safe = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
with torch.no_grad():
    x_safe.add_(1.0)  # OK: no graph exists yet
y_safe = (x_safe ** 2).sum()
y_safe.backward()
print(f"(1) Safe in-place on leaf (no graph yet): x_safe.grad = {x_safe.grad}")

# ── (2) Dangerous: in-place on a leaf AFTER creating a dependent tensor ───────
print("\n(2) In-place on leaf used in a graph -> RuntimeError:")
x_bad = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_bad = x_bad * 2.0   # x_bad now has a dependent node
try:
    x_bad.add_(1.0)   # modifies x_bad in-place after graph was created
    y_bad.sum().backward()
except RuntimeError as exc:
    print(f"  RuntimeError: {str(exc)[:80]}...")

# ── (3) Dangerous: .data bypass (no error, but WRONG gradients) ───────────────
print("\n(3) .data bypass — no error but incorrect gradients:")
x_data = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_data = x_data * 2.0
x_data.data.add_(1.0)   # bypasses version counter — PyTorch does NOT catch this
y_data.sum().backward()  # gradient uses OLD x_data values
print(f"  x_data.grad = {x_data.grad}  (computed using stale data — WRONG!)")
print(f"  Correct grad should be [2., 2., 2.]  (d/dx of 2x at any x)")

# ── (4) Safe alternative: use out-of-place operations ─────────────────────────
print("\n(4) Correct pattern — out-of-place:")
x_ok = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y_ok = (x_ok + 1.0) * 2.0   # out-of-place: x_ok is never modified
y_ok.sum().backward()
print(f"  x_ok.grad = {x_ok.grad}  (correct: d/dx of 2*(x+1) = 2)")

print("\nRule: never apply in-place ops to tensors with requires_grad=True or grad_fn.")


### 3.3 Higher-Order Gradients

By default, PyTorch frees the intermediate computation graph during `.backward()`.
Passing `create_graph=True` to `torch.autograd.grad()` retains the graph so that
**gradients of gradients** can be computed.

Applications:
- **MAML** (meta-learning): differentiates through an inner optimisation loop.
- **Gradient penalty** in GANs: penalises $\|\nabla_x D(x)\|^2$.
- **Hessian-vector products** in second-order optimisers.


In [ ]:
# ── Second-order gradients ────────────────────────────────────────────────────
print("=== Second-order gradients via create_graph=True ===")

# f(x) = x^4  =>  f'(x) = 4x^3  =>  f''(x) = 12x^2
x_ho = torch.tensor([2.0], requires_grad=True)
y_ho = x_ho ** 4

dy_dx = torch.autograd.grad(
    y_ho, x_ho, create_graph=True   # retain graph for second backward
)[0]
print(f"  dy/dx  at x=2: {dy_dx.item():.1f}   (expected {4 * 2**3:.1f})")

d2y_dx2 = torch.autograd.grad(dy_dx, x_ho)[0]
print(f"  d2y/dx2 at x=2: {d2y_dx2.item():.1f}  (expected {12 * 2**2:.1f})")

# ── Gradient penalty (GAN-style): ||grad_x D(x)||^2 ──────────────────────────
print("\n=== Gradient penalty ===")
discriminator = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
x_gp = torch.randn(16, 4, requires_grad=True)
d_x  = discriminator(x_gp).sum()

grad_d = torch.autograd.grad(
    d_x, x_gp, create_graph=True
)[0]                             # shape: (16, 4)
gradient_penalty = (grad_d.norm(dim=1) ** 2).mean()
gradient_penalty.backward()      # second-order backward
print(f"  Gradient penalty: {gradient_penalty.item():.4f}")
print(f"  x_gp.grad is not None: {x_gp.grad is not None}")
print("  (Second backward propagated through the gradient norm computation.)")

# ── Hessian-vector product (HVP) ──────────────────────────────────────────────
print("\n=== Hessian-vector product (HVP) ===")
# HVP: H*v = d/dw [grad_w L . v]  (one backward call per vector)
w_hvp = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss_hvp = (w_hvp[0]**2 + 2*w_hvp[1]**2 + 3*w_hvp[2]**2)
# H = diag([2, 4, 6]) for this quadratic

v_hvp = torch.tensor([1.0, 0.0, 0.0])   # direction vector
grad_w = torch.autograd.grad(loss_hvp, w_hvp, create_graph=True)[0]
hvp    = torch.autograd.grad(grad_w, w_hvp, grad_outputs=v_hvp)[0]
print(f"  HVP (H@v for v=[1,0,0]): {hvp}  (expected [2., 0., 0.] = 2*w_hvp[0]*v[0])")

# ── Hessian diagonal via double backward ──────────────────────────────────────
# For f(w) = w0^2 + 2*w1^2 + 3*w2^2, H = diag([2, 4, 6]).
print("\n=== Hessian diagonal via double backward ===")
w_hess = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
f_hess = w_hess[0]**2 + 2.0*w_hess[1]**2 + 3.0*w_hess[2]**2
grad_h = torch.autograd.grad(f_hess, w_hess, create_graph=True)[0]
h_diag = torch.stack([
    torch.autograd.grad(grad_h[i], w_hess, retain_graph=(i < 2))[0][i]
    for i in range(3)
])
print(f"  Hessian diagonal: {h_diag.tolist()}  (expected [2., 4., 6.])")
print("  Full Hessian of an n-param model requires n backward passes (expensive).")
print("  In practice, diagonal approximations (Adagrad, Adam) are used instead.")


### 3.4 Jacobians and Jacobian-Vector Products

For a vector-valued function $f: \mathbb{R}^n \to \mathbb{R}^m$, the
**Jacobian** $J \in \mathbb{R}^{m \times n}$ has $J_{ij} = \partial f_i / \partial x_j$.

- **VJP** (vector-Jacobian product): $v^\top J$ — this is what `.backward()` computes
  (reverse-mode AD, efficient when $m \ll n$).
- **JVP** (Jacobian-vector product): $J v$ — forward-mode AD, efficient when $n \ll m$.
- **Full Jacobian**: computed by calling backward $m$ times (one per output).


In [ ]:
# ── Full Jacobian via torch.autograd.functional.jacobian ──────────────────────
print("=== Full Jacobian ===")

def f_vec(x: torch.Tensor) -> torch.Tensor:
    '''Map R^3 -> R^3: f(x) = [x0^2 + x1, x0*x1 + x2^2, x1 + x2].

    Args:
        x: Input tensor of shape (3,).

    Returns:
        Output tensor of shape (3,).
    '''
    return torch.stack([
        x[0] ** 2 + x[1],
        x[0] * x[1] + x[2] ** 2,
        x[1] + x[2],
    ])


x_jac = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
J_auto = torch.autograd.functional.jacobian(f_vec, x_jac)
print(f"  Jacobian shape: {J_auto.shape}")   # (3, 3)

# Analytical Jacobian at x=[1,2,3]:
#   df1/dx: [2*1, 1, 0]  = [2, 1, 0]
#   df2/dx: [x1, x0, 2*x2] = [2, 1, 6]
#   df3/dx: [0, 1, 1]
J_analytic = torch.tensor([
    [2.0, 1.0, 0.0],
    [2.0, 1.0, 6.0],
    [0.0, 1.0, 1.0],
])
print(f"  Autograd Jacobian:")
print(f"  {J_auto.numpy()}")
print(f"  Analytical Jacobian:")
print(f"  {J_analytic.numpy()}")
print(f"  Max diff: {(J_auto - J_analytic).abs().max().item():.2e}")

# ── VJP (reverse-mode, standard backward) ─────────────────────────────────────
print("\n=== VJP: v^T @ J (reverse-mode) ===")
v_vjp = torch.ones(3)
y_vjp = f_vec(x_jac)
vjp   = torch.autograd.grad(y_vjp, x_jac, grad_outputs=v_vjp)[0]
expected_vjp = J_analytic.T @ v_vjp   # column sums of J
print(f"  VJP autograd: {vjp}")
print(f"  VJP analytic: {expected_vjp}")

# ── JVP (forward-mode via torch.autograd.functional.jvp) ─────────────────────
print("\n=== JVP: J @ v (forward-mode) ===")
v_jvp = torch.ones(3)
_, jvp_result = torch.autograd.functional.jvp(f_vec, x_jac, v=v_jvp)
expected_jvp  = J_analytic @ v_jvp   # row sums of J
print(f"  JVP autograd: {jvp_result}")
print(f"  JVP analytic: {expected_jvp}")
print(f"  Max diff: {(jvp_result - expected_jvp).abs().max().item():.2e}")

# ── Efficiency comparison: VJP vs full Jacobian ───────────────────────────────
print("\n=== Efficiency: VJP vs full Jacobian ===")
n_in, n_out = 50, 100
W_eff = torch.randn(n_out, n_in, requires_grad=True)
x_eff = torch.randn(n_in,  requires_grad=True)

t0 = time.perf_counter()
for _ in range(200):
    y_eff = W_eff @ x_eff
    vjp_eff = torch.autograd.grad(y_eff, x_eff, grad_outputs=torch.ones(n_out))[0]
t_vjp = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(200):
    y_eff2 = W_eff @ x_eff
    J_eff  = torch.autograd.functional.jacobian(lambda x: W_eff @ x, x_eff)
t_jac = time.perf_counter() - t0

print(f"  VJP ({n_in}x{n_out}):          {t_vjp*1000:.1f} ms (200 iters)")
print(f"  Full Jacobian ({n_in}x{n_out}): {t_jac*1000:.1f} ms (200 iters)")
print(f"  Ratio: {t_jac/t_vjp:.1f}x  (Full Jacobian = n_out VJP calls)")

# ── Batched Jacobian: per-sample Jacobians over a mini-batch ──────────────────
# In practice we often want J_i = df(x_i)/dx_i independently for each sample.
# PyTorch has torch.vmap for this, but a clear loop is equivalent for small batches.
print("\n=== Batched Jacobian (loop over batch) ===")

def batch_jacobian(
    fn: Callable,
    x_batch: torch.Tensor,
) -> torch.Tensor:
    '''Compute the Jacobian of fn independently for each sample in x_batch.

    Args:
        fn: Function mapping (n_in,) -> (n_out,) tensor.
        x_batch: Input batch of shape (batch_size, n_in).

    Returns:
        Jacobian tensor of shape (batch_size, n_out, n_in).
    '''
    J_list: list[torch.Tensor] = []
    for i in range(x_batch.shape[0]):
        x_i = x_batch[i].detach().requires_grad_(True)
        J_i = torch.autograd.functional.jacobian(fn, x_i)
        J_list.append(J_i)
    return torch.stack(J_list)


x_bj = torch.randn(6, 3)
J_bj = batch_jacobian(f_vec, x_bj)
print(f"  Batched Jacobian shape: {J_bj.shape}")   # (6, 3, 3)
fnorm_per_sample = J_bj.norm(dim=(1, 2))
print(f"  Frobenius norm per sample: {fnorm_per_sample.tolist()}")
print(f"  Mean Frobenius norm: {fnorm_per_sample.mean().item():.4f}")

# ── Input sensitivity: ||J||_F as a proxy for output sensitivity to input ─────
print("\n=== Input sensitivity analysis ===")
# A large Jacobian norm means the output is highly sensitive to changes in input.
# We measure sensitivity across 100 random inputs.
n_sens = 100
x_sens = torch.randn(n_sens, 3)
J_sens = batch_jacobian(f_vec, x_sens)
fnorms_sens = J_sens.norm(dim=(1, 2)).numpy()
print(f"  Sensitivity (||J||_F) across {n_sens} random inputs:")
print(f"    mean={fnorms_sens.mean():.4f}  std={fnorms_sens.std():.4f}"
      f"  min={fnorms_sens.min():.4f}  max={fnorms_sens.max():.4f}")

fig_sens, ax_sens = plt.subplots(figsize=(7, 3))
ax_sens.hist(fnorms_sens, bins=20, color="#1f77b4", edgecolor="k", density=True)
ax_sens.set_xlabel("||J||_F (Jacobian Frobenius norm)", fontsize=11)
ax_sens.set_ylabel("Density", fontsize=11)
ax_sens.set_title("Input Sensitivity Distribution over 100 Random Inputs",
                   fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print("  Large ||J||_F -> output changes rapidly with input (sensitive region).")


---
## Part 4 — Training with Custom Autograd Functions

We now build a complete MLP training pipeline using our custom `CustomReLUFn`
activation.  The `CustomMLP` model is architecturally identical to `StandardMLP`
(which uses `torch.relu`), but every activation step routes through our
hand-written `backward`.

The expected outcome: **identical training loss curves** — since `CustomReLUFn`
computes exactly the same forward and backward as `torch.relu`.  The `grad_fn`
chain in the computation graph will show `CustomReLUFnBackward` instead of
`ReluBackward`.


In [ ]:
# ── Models: CustomMLP (our Function) vs StandardMLP (torch.relu) ─────────────

class CustomMLP(nn.Module):
    '''Fully-connected MLP using CustomReLUFn for hidden activations.

    Attributes:
        linears: ModuleList of nn.Linear layers.
        n_hidden: Number of hidden layers (len(linears) - 1).
    '''

    def __init__(
        self,
        layer_sizes: list[int],
    ) -> None:
        '''Build the MLP as a list of linear layers.

        Args:
            layer_sizes: List [n_in, h1, ..., n_out] of layer widths.
        '''
        super().__init__()
        self.linears  = nn.ModuleList([
            nn.Linear(fan_in, fan_out)
            for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:])
        ])
        self.n_hidden = len(self.linears) - 1

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        '''Forward pass using CustomReLUFn for hidden activations.

        Args:
            x: Input tensor of shape (batch_size, n_in).

        Returns:
            Output tensor of shape (batch_size, n_out).
        '''
        for i, linear in enumerate(self.linears):
            x = linear(x)
            if i < self.n_hidden:
                x = custom_relu(x)     # our custom Function
        return x


class StandardMLP(nn.Module):
    '''Identical architecture to CustomMLP, but uses torch.relu.

    Attributes:
        linears: ModuleList of nn.Linear layers.
        n_hidden: Number of hidden layers.
    '''

    def __init__(
        self,
        layer_sizes: list[int],
    ) -> None:
        '''Build the MLP as a list of linear layers.

        Args:
            layer_sizes: List [n_in, h1, ..., n_out] of layer widths.
        '''
        super().__init__()
        self.linears  = nn.ModuleList([
            nn.Linear(fan_in, fan_out)
            for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:])
        ])
        self.n_hidden = len(self.linears) - 1

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        '''Forward pass using torch.relu for hidden activations.

        Args:
            x: Input tensor of shape (batch_size, n_in).

        Returns:
            Output tensor of shape (batch_size, n_out).
        '''
        for i, linear in enumerate(self.linears):
            x = linear(x)
            if i < self.n_hidden:
                x = torch.relu(x)      # built-in relu
        return x


def train_mlp(
    model: nn.Module,
    train_ldr: DataLoader,
    val_ldr: DataLoader,
    num_epochs: int,
    lr: float,
    loss_fn: Callable,
) -> tuple[list[float], list[float]]:
    '''Train an MLP for regression and return per-epoch losses.

    Args:
        model: PyTorch Module to train.
        train_ldr: Training DataLoader.
        val_ldr: Validation DataLoader.
        num_epochs: Number of training epochs.
        lr: Adam learning rate.
        loss_fn: Callable loss function (y_hat, y_true) -> scalar tensor.

    Returns:
        Tuple of (train_losses, val_losses) per epoch.
    '''
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_losses: list[float] = []
    val_losses:   list[float] = []

    for epoch in range(num_epochs):
        model.train()
        ep_loss = 0.0
        n_batches = 0
        t_start = time.time()
        for X_b, y_b in train_ldr:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            y_hat = model(X_b)
            loss  = loss_fn(y_hat, y_b)
            loss.backward()
            optimizer.step()
            ep_loss   += loss.item()
            n_batches += 1
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_vb, y_vb in val_ldr:
                X_vb, y_vb = X_vb.to(device), y_vb.to(device)
                val_loss += loss_fn(model(X_vb), y_vb).item()
        tr_avg = ep_loss / n_batches
        vl_avg = val_loss / len(val_ldr)
        train_losses.append(tr_avg)
        val_losses.append(vl_avg)
        elapsed = time.time() - t_start
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:>2d}/{num_epochs} | "
                  f"Train Loss: {tr_avg:.4f} | Val Loss: {vl_avg:.4f} | "
                  f"Val RMSE: {float(np.sqrt(vl_avg)):.4f} | Time: {elapsed:.1f}s")
    return train_losses, val_losses


arch = [N_FEATURES, HIDDEN_DIM, HIDDEN_DIM, OUTPUT_DIM]

torch.manual_seed(SEED)
custom_model = CustomMLP(layer_sizes=arch)
print("=== CustomMLP (CustomReLUFn) ===")
cust_tr_losses, cust_val_losses = train_mlp(
    custom_model, train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE,
    loss_fn=nn.MSELoss(),
)

torch.manual_seed(SEED)
std_model = StandardMLP(layer_sizes=arch)
print("\n=== StandardMLP (torch.relu) ===")
std_tr_losses, std_val_losses = train_mlp(
    std_model, train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE,
    loss_fn=nn.MSELoss(),
)


In [ ]:
# ── Training curve comparison & graph inspection ─────────────────────────────
fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(13, 4))

epochs_ax = list(range(1, NUM_EPOCHS + 1))
axes_cmp[0].plot(epochs_ax, cust_tr_losses, color="#1f77b4", lw=2,
                 label="CustomMLP train")
axes_cmp[0].plot(epochs_ax, cust_val_losses, color="#1f77b4", lw=2, ls="--",
                 label="CustomMLP val")
axes_cmp[0].plot(epochs_ax, std_tr_losses,   color="#d62728", lw=2,
                 label="StandardMLP train")
axes_cmp[0].plot(epochs_ax, std_val_losses,  color="#d62728", lw=2, ls="--",
                 label="StandardMLP val")
axes_cmp[0].set_xlabel("Epoch", fontsize=11)
axes_cmp[0].set_ylabel("MSE Loss", fontsize=11)
axes_cmp[0].set_title("Loss: CustomReLUFn vs torch.relu", fontsize=11, fontweight="bold")
axes_cmp[0].legend(fontsize=9)
axes_cmp[0].grid(alpha=0.3)

final_diff = abs(cust_val_losses[-1] - std_val_losses[-1])
axes_cmp[1].bar(["CustomMLP\n(CustomReLUFn)", "StandardMLP\n(torch.relu)"],
                [cust_val_losses[-1], std_val_losses[-1]],
                color=["#1f77b4", "#d62728"], edgecolor="k", lw=0.8)
axes_cmp[1].set_ylabel("Final Val MSE", fontsize=11)
axes_cmp[1].set_title(f"Final Val MSE (diff = {final_diff:.6f})",
                       fontsize=11, fontweight="bold")
axes_cmp[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# ── grad_fn chain inspection: different types but same computation ─────────────
print("=== grad_fn comparison (single forward batch) ===")
x_ins = next(iter(train_loader))[0][:4].to(device)
out_cust = custom_model(x_ins)
out_std  = std_model(x_ins)
print(f"CustomMLP  output grad_fn: {out_cust.grad_fn}")
print(f"StandardMLP output grad_fn: {out_std.grad_fn}")

print("\n=== First hidden activation grad_fn ===")
# Access the intermediate output after first linear + relu
with torch.no_grad():
    h_cust = custom_relu(custom_model.linears[0](x_ins).detach().requires_grad_(True))
    h_std  = torch.relu( std_model.linears[0](   x_ins).detach().requires_grad_(True))
print(f"custom_relu output grad_fn: {h_cust.grad_fn}")
print(f"torch.relu  output grad_fn: {h_std.grad_fn}")
print("Different grad_fn TYPE but same mathematical operation and gradient.")

# ── Summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    "Model":          ["CustomMLP (CustomReLUFn)", "StandardMLP (torch.relu)"],
    "Final Train MSE": [cust_tr_losses[-1], std_tr_losses[-1]],
    "Final Val MSE":   [cust_val_losses[-1], std_val_losses[-1]],
    "gradcheck pass":  ["yes", "yes (verified by PyTorch)"],
})
print("\n", results_df.to_string(index=False, float_format="{:.5f}".format))
print("\nIdentical losses confirm that CustomReLUFn and torch.relu are numerically equivalent.")


### 4.2 Gradient Norm Heatmap Across Training

We re-train `CustomMLP` from scratch and record the per-epoch average
gradient norm $\|\partial\mathcal{L}/\partial\mathbf{W}_l\|_F$ for each layer.
This produces a 2-D heatmap (epochs × layers) that visualises **how gradient
magnitude evolves** over training — a practical diagnostic for learning dynamics.


In [ ]:
# ── Gradient norm heatmap: weight matrix norms across layers and epochs ────────
# Re-train a fresh model and record the gradient norm of each layer's weight
# matrix at the end of every epoch.

torch.manual_seed(SEED)
hm_model = CustomMLP(layer_sizes=arch).to(device)
hm_opt   = optim.Adam(hm_model.parameters(), lr=LEARNING_RATE)
hm_crit  = nn.MSELoss()

# grad_norms_hm[epoch][layer_idx] = ||dL/dW||_F
NUM_HM_EPOCHS = NUM_EPOCHS
grad_norms_hm: list[list[float]] = []

for epoch in range(NUM_HM_EPOCHS):
    hm_model.train()
    ep_norms: list[float] = [0.0] * len(hm_model.linears)
    n_batches = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        hm_opt.zero_grad()
        loss_hm = hm_crit(hm_model(X_b), y_b)
        loss_hm.backward()
        for lyr_idx, linear in enumerate(hm_model.linears):
            if linear.weight.grad is not None:
                ep_norms[lyr_idx] += float(linear.weight.grad.norm().item())
        hm_opt.step()
        n_batches += 1
    grad_norms_hm.append([n / n_batches for n in ep_norms])

# Convert to numpy array: shape (NUM_EPOCHS, n_layers)
norms_array = np.array(grad_norms_hm)   # (epochs, layers)

fig_hm, ax_hm = plt.subplots(figsize=(10, 4))
im_hm = ax_hm.imshow(norms_array.T, aspect="auto", cmap="YlOrRd", origin="lower")
ax_hm.set_xlabel("Epoch", fontsize=11)
ax_hm.set_ylabel("Layer index (0 = input-side)", fontsize=11)
ax_hm.set_yticks(range(len(hm_model.linears)))
ax_hm.set_yticklabels([
    f"L{i+1} ({arch[i]}->{arch[i+1]})" for i in range(len(hm_model.linears))
], fontsize=9)
ax_hm.set_title("Gradient Norm ||dL/dW||_F per Layer per Epoch (CustomMLP)",
                  fontsize=11, fontweight="bold")
plt.colorbar(im_hm, ax=ax_hm, label="||dL/dW||_F")
plt.tight_layout()
plt.show()

print("Gradient norm heatmap summary:")
for lyr_idx in range(len(hm_model.linears)):
    mean_n = float(norms_array[:, lyr_idx].mean())
    print(f"  Layer {lyr_idx + 1} ({arch[lyr_idx]:>3d}->{arch[lyr_idx+1]:>3d}): "
          f"mean norm = {mean_n:.5f}")

# Compare first vs last layer norm ratio across training
first_layer_norms = norms_array[:, 0]
last_layer_norms  = norms_array[:, -1]
ratio_series = last_layer_norms / (first_layer_norms + 1e-12)
print(f"\nMean ratio (last/first layer grad norm): {ratio_series.mean():.2f}")
print("Ratio > 1 means the output layer has larger gradients than the input layer.")
print("This is the precursor to vanishing gradients in deeper networks.")
epochs_increasing = int((np.diff(norms_array[:, 0]) > 0).sum())
print(f"First-layer epochs with increasing grad norm: {epochs_increasing}/{NUM_HM_EPOCHS - 1}")
print("Stable or decreasing norms = learning rate is appropriate for the task.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Every operation builds a graph node:** When `requires_grad=True`, PyTorch
   records a `grad_fn` for every tensor produced by an operation.  `.backward()`
   traverses this DAG in reverse, applying the chain rule at each node.

2. **Leaf vs non-leaf determines where `.grad` is stored:** Only leaf tensors
   (user-created with `requires_grad=True`) accumulate `.grad`.  Use
   `retain_grad()` or `register_hook()` to inspect intermediate values.

3. **Custom `Function` exposes full control:** By implementing `forward` and
   `backward` as static methods, you can introduce any differentiable (or
   surrogate-differentiable) operation into the PyTorch graph.
   Always verify with `torch.autograd.gradcheck`.

4. **`no_grad()` and `detach()` control gradient scope:**
   - `torch.no_grad()` disables graph construction entirely (fast, memory-efficient inference).
   - `.detach()` cuts a tensor out of the graph, acting as a stop-gradient operator.
   - Avoid in-place operations on tensors with `grad_fn` — they corrupt the graph.

5. **Higher-order gradients require `create_graph=True`:** Passing this flag
   retains the backward graph so that `torch.autograd.grad` can differentiate
   through it again.  Essential for MAML, gradient penalties, and Hessian-based
   methods.

---

### What's Next

- **05-08 — Weight Initialisation:** Xavier, He/Kaiming, orthogonal — how the
  initial gradient flow depends on weight scale.
- **05-09 — Optimisers:** SGD, Momentum, Adam, AdamW — each defines a different
  rule for converting gradients into parameter updates.
